# GLIDE — Quickstart

**GLIDE** turns a small set of true evaluation labels (e.g. human annotations) + a large pool of proxy evaluation labels (e.g. LLM-as-judge annotations) into a **bias-corrected metric with a valid confidence interval**.

| Ingredient | Role |
|-----------|------|
| True labels | Ground truth — accurate but slow and expensive |
| Proxy labels | Proxy — cheap and fast but biased |

**The full workflow fits in three lines:**

```python
dataset = Dataset(records)
result  = PPIMeanEstimator().estimate(dataset, y_true_field="y_true", y_proxy_field="y_proxy")
print(result.summary())
```

## Installation

Before running the code below (see bottom for complete notebook), install **glide-py**:

```bash
pip install glide-py
```

In [2]:
from glide.core.simulated import generate_dataset_binary  # for the demo below
from glide.estimators.ppi import PPIMeanEstimator

## Step 1 — Assemble Your Data

A `Dataset` is a list of Python dicts. Records come in two kinds:

- **Labeled** — have both `y_true` and `y_proxy`
- **Unlabeled** — have only `y_proxy`

Any key names work, pass them to `estimate()` at call time.

```python
labeled   = [{"y_true": 1, "y_proxy": 0}, {"y_true": 0, "y_proxy": 0}]
unlabeled = [{"y_proxy": 0}, {"y_proxy": 1}]
dataset   = Dataset(labeled + unlabeled)
```

To run this notebook end-to-end, generate a synthetic dataset with the built-in helper:

In [4]:
# Simulate: 100 groundtruth + 900 proxy-labeled conversations
# True rate: 15%  |  Proxy rate: 10% (biased low)
dataset = generate_dataset_binary(
    n=100,            # labeled records
    N=900,            # unlabeled records
    true_mean=0.15,   # true rate
    proxy_mean=0.10,  # proxy rate (biased)
    correlation=0.70,
    random_seed=0,
)

print(f"Dataset : {len(dataset)} records")
print(f"  labeled   : {sum(1 for r in dataset if 'y_true' in r)}")
print(f"  unlabeled : {sum(1 for r in dataset if 'y_true' not in r)}")
print()
print(f"Labeled sample   -> {dataset[0]}")
print(f"Unlabeled sample -> {dataset[-1]}")

Dataset : 1000 records
  labeled   : 100
  unlabeled : 900

Labeled sample   -> {'y_true': 0, 'y_proxy': 0}
Unlabeled sample -> {'y_proxy': 0}


## Step 2 — Estimate the Metric

`PPIMeanEstimator.estimate()` corrects the proxy's bias using the labeled subset, then applies the correction across the full dataset.

In [5]:
result = PPIMeanEstimator().estimate(
    dataset,
    y_true_field="y_true",
    y_proxy_field="y_proxy",
    metric_name="Evaluated metric", # e.g. an LLM's output Hallucination rate
    confidence_level=0.95,
)

print(result.summary())

Metric: Evaluated metric
Point Estimate: 0.159
Confidence Interval (95%): [0.11, 0.21]
Estimator : PPIMeanEstimator
n_true: 100
n_proxy: 1000


The effective sample size is the number of true labels needed to achieve the same confidence without PPI.

## Step 3 — Read the Result

`InferenceResult` exposes the point estimate, confidence interval, and a built-in z-test.

| Attribute | Description |
|-----------|-------------|
| `result.result.mean` | Bias-corrected point estimate |
| `result.result.lower_bound` / `upper_bound` | Confidence interval bounds |
| `result.result.std` | Standard error |
| `result.result.test_null_hypothesis(h0, alternative)` | One-sample z-test |

In [6]:
# Point estimate and confidence interval
print(f"Estimate : {result.result.mean:.1%}")
print(f"95% CI   : [{result.result.lower_bound:.1%}, {result.result.upper_bound:.1%}]")
print(f"Std err  : {result.result.std:.4f}")
print()

# Test H0: hallucination rate = 10% (the judge's claim)
z, p, _ = result.result.test_null_hypothesis(
    h0_value=0.10,
    alternative="larger",  # H1: true rate > 10%
)
print("H0: rate = 10%  |  H1: rate > 10%")
print(f"z = {z:.2f}  |  p = {p:.4f}")
print("Reject H0 — the rate is significantly above 10%." if p < 0.05 else "Cannot reject H0 at the 5% level.")

Estimate : 15.9%
95% CI   : [10.5%, 21.2%]
Std err  : 0.0273

H0: rate = 10%  |  H1: rate > 10%
z = 2.15  |  p = 0.0156
Reject H0 — the rate is significantly above 10%.


## Using Real Data

Replace `generate_dataset_binary` with any list of dicts. Labeled records must have both `y_true` and `y_proxy`; unlabeled records need only `y_proxy`.

```python
import json
import pandas as pd

# From a pandas DataFrame
records = df.to_dict(orient="records")

# From a JSON file
with open("annotations.json") as f:
    records = json.load(f)

dataset = Dataset(records)

```

Then you can define an estimator (e.g. PPI) and pass your actual column names by providing values for `y_true_field` and `y_proxy_field` in the snippet below.

```
ppi_estimator = PPIMeanEstimator()
result = ppi_estimator.estimate(
    dataset,
    y_true_field=...,
    y_proxy_field=...,
    metric_name="Evaluated metric", # e.g. an LLM's output Hallucination rate
    confidence_level=0.95,
)

print(result.summary())
```



You can download this notebook [here](https://github.com/EmertonData/glide/raw/refs/heads/main/docs/examples/quickstart.ipynb)